# Notebook 03 — Supervised Classification

**Question.** Which classifier family best separates UNSW-NB15's ten `attack_cat` classes on
training data alone, how much of that score comes from the `sttl`/`ct_state_ttl` testbed
shortcut, and what would a SOC actually deploy once the label-noise floor and a cost-based
alert threshold are taken into account?

**The test partition is spent once.** Every hyperparameter is selected and every model is
chosen using training data only (`RandomizedSearchCV` over `StratifiedKFold`, and out-of-fold
`cross_val_predict` for the binary threshold). The cleaned testing partition is scored exactly
one time, after every model has been finally refit — never during model selection.

In [1]:
import json
from dataclasses import dataclass
from time import perf_counter
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from nids import paths
from nids.columns import TARGET_COLUMNS, feature_columns
from nids.data import load_clean_partitions
from nids.preprocessing import build_preprocessor
from nids.results import ResultsWriter
from nids.sampling import stratified_subsample

# Constants (design `c01`)
SEED = 42
CV_FOLDS = 3
N_ITER_MAX = 15
SEARCH_ROWS = 30_000     # stratified TRAIN subsample used by every search
GP_MAX_ROWS = 2_000      # O(n^3) time, O(n^2) memory, one fit per class
SVC_MAX_ROWS = 5_000     # O(n^2..n^3)
KNN_REF_ROWS = 20_000    # O(n_train x n_test) at predict time
COST_FN, COST_FP = 20.0, 1.0
SUBSAMPLE_FLOOR = 50
VARIANT_COLORS = {"with_ttl": "#2E5A88", "without_ttl": "#C0603A"}

np.random.seed(SEED)
print("Constants loaded.")

Constants loaded.


## Compute budget

| Cap | Value |
|---|---|
| Wall clock | under ~15 minutes on 16 threads |
| `n_jobs` | -1 wherever supported (RandomForest search: outer -1 / inner 1; HistGradientBoosting search: outer 4) |
| `RandomizedSearchCV` | `n_iter <= 15`, `StratifiedKFold(n_splits=3)`, on a 30,000-row stratified train subsample |
| `GaussianProcessClassifier` | at most 2,000 rows, fixed kernel, no search |
| `SVC(RBF)` | at most 5,000 rows |
| `KNeighborsClassifier` | at most 20,000 reference rows |
| Every subsample | `stratified_subsample`, seed 42, floor 50, so rare classes (e.g. `Worms`) survive |

In [2]:
writer = ResultsWriter("classification")
t_notebook = perf_counter()
print(f"Writing results under {writer.directory}")

Writing results under /home/pato/Desktop/machine_learning_project/results/classification


In [3]:
cleaned = load_clean_partitions()

print(f"train: {cleaned.train.shape}")
print(f"test:  {cleaned.test.shape}")
print("train attack_cat counts:")
print(cleaned.train["attack_cat"].value_counts())
print("test attack_cat counts:")
print(cleaned.test["attack_cat"].value_counts())

writer.add_metric("n_train_rows", int(len(cleaned.train)), "Rows in the cleaned training partition.")
writer.add_metric("n_test_rows", int(len(cleaned.test)), "Rows in the cleaned testing partition.")

train: (107740, 41)
test:  (78084, 41)
train attack_cat counts:
attack_cat
Normal            51890
Exploits          19844
Fuzzers           16150
Reconnaissance     7522
Generic            4181
DoS                3806
Analysis           1594
Backdoor           1535
Shellcode          1091
Worms               127
Name: count, dtype: int64
test attack_cat counts:
attack_cat
Normal            35806
Generic           17176
Exploits          10954
Fuzzers            6018
DoS                3887
Reconnaissance     2592
Analysis            663
Backdoor            579
Shellcode           365
Worms                44
Name: count, dtype: int64


In [4]:
# Assertion 1: attack_cat/label never enter the feature allow-list.
feat_with = feature_columns(True)
feat_without = feature_columns(False)

assert set(TARGET_COLUMNS).isdisjoint(feat_with), "with_ttl feature set leaked a target column"
assert set(TARGET_COLUMNS).isdisjoint(feat_without), "without_ttl feature set leaked a target column"
assert len(feat_with) == 39, f"expected 39 with_ttl features, got {len(feat_with)}"
assert len(feat_without) == 37, f"expected 37 without_ttl features, got {len(feat_without)}"

writer.add_metric("n_features_with_ttl", len(feat_with), "Feature count with sttl/ct_state_ttl included.")
writer.add_metric("n_features_without_ttl", len(feat_without), "Feature count with sttl/ct_state_ttl excluded.")

print(f"Assertion 1 passed: target columns never enter the feature allow-list ({len(feat_with)} / {len(feat_without)} features).")

Assertion 1 passed: target columns never enter the feature allow-list (39 / 37 features).


In [5]:
# The test-fit guard: every .fit call in this notebook is routed through
# fit_train_only(). It raises (and counts) if the frame it receives
# fingerprints as the test partition -- shape and column order both have to
# match, and no legitimate training call in this notebook (full train, a
# stratified subsample, or a column-selected slice of either) shares the test
# partition's row count, so this never false-positives while catching a real
# leak immediately.
TEST_FIT_COUNT = 0
TRAIN_FRAME_ID = id(cleaned.train)
TEST_FRAME_FINGERPRINT = (cleaned.test.shape, tuple(cleaned.test.columns))


def fit_train_only(estimator, X, y=None):
    """Fit `estimator` on (X, y); raise and count if X fingerprints as the test partition."""
    global TEST_FIT_COUNT
    fingerprint = (X.shape, tuple(X.columns)) if isinstance(X, pd.DataFrame) else None
    if fingerprint == TEST_FRAME_FINGERPRINT:
        TEST_FIT_COUNT += 1
        raise RuntimeError(
            "fit_train_only refused to fit: X's shape and columns match the "
            "test partition's fingerprint."
        )
    if y is None:
        estimator.fit(X)
    else:
        estimator.fit(X, y)
    return estimator


print(f"TEST_FIT_COUNT initialised to {TEST_FIT_COUNT}; guarding fingerprint shape {TEST_FRAME_FINGERPRINT[0]}")

TEST_FIT_COUNT initialised to 0; guarding fingerprint shape (78084, 41)


In [6]:
search_sub = stratified_subsample(cleaned.train, stratify_by="attack_cat", max_rows=SEARCH_ROWS, floor=SUBSAMPLE_FLOOR, seed=SEED)
gp_sub = stratified_subsample(cleaned.train, stratify_by="attack_cat", max_rows=GP_MAX_ROWS, floor=SUBSAMPLE_FLOOR, seed=SEED)
svc_sub = stratified_subsample(cleaned.train, stratify_by="attack_cat", max_rows=SVC_MAX_ROWS, floor=SUBSAMPLE_FLOOR, seed=SEED)
knn_sub = stratified_subsample(cleaned.train, stratify_by="attack_cat", max_rows=KNN_REF_ROWS, floor=SUBSAMPLE_FLOOR, seed=SEED)

subsample_allocations = pd.concat(
    [
        search_sub.to_frame().assign(subsample="search_sub"),
        gp_sub.to_frame().assign(subsample="gp_sub"),
        svc_sub.to_frame().assign(subsample="svc_sub"),
        knn_sub.to_frame().assign(subsample="knn_sub"),
    ],
    ignore_index=True,
)
writer.add_table(
    subsample_allocations,
    name="subsample_allocations",
    title="Stratified subsample class allocations",
    description="Per-class realized allocation (available, allocated, proportional, floor_applied) for each of the four stratified subsamples used by this notebook.",
    sort_by=["subsample", "label"],
)

writer.add_metric("search_subsample_rows", len(search_sub.data), "Rows in the 30,000-row stratified train subsample used by every hyperparameter search.")
writer.add_metric("gp_training_rows", len(gp_sub.data), "Rows GaussianProcessClassifier is trained on (capped at GP_MAX_ROWS).")
writer.add_metric("svc_training_rows", len(svc_sub.data), "Rows SVC(RBF) is trained on (capped at SVC_MAX_ROWS).")
writer.add_metric("knn_reference_rows", len(knn_sub.data), "Rows KNeighborsClassifier references at predict time (capped at KNN_REF_ROWS).")

print(f"search_sub={len(search_sub.data)} gp_sub={len(gp_sub.data)} svc_sub={len(svc_sub.data)} knn_sub={len(knn_sub.data)}")

search_sub=30000 gp_sub=2000 svc_sub=5000 knn_sub=20000


In [7]:
def load_label_noise_floor() -> dict[str, Any]:
    """Read the label-noise floor from results/data_cleaning/manifest.json, or degrade gracefully."""
    manifest_path = paths.results_root() / "data_cleaning" / "manifest.json"
    if not manifest_path.is_file():
        return {"available": False, "rows": "unavailable", "share": "unavailable"}
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    metrics = {entry["name"]: entry["value"] for entry in manifest.get("metrics", [])}
    rows = metrics.get("test_contradictory_rows_retained", "unavailable")
    share = metrics.get("test_contradictory_share_of_retained", "unavailable")
    return {"available": True, "rows": rows, "share": share}


LABEL_NOISE_FLOOR = load_label_noise_floor()
print(f"label noise floor: {LABEL_NOISE_FLOOR}")

dataset_shapes = pd.DataFrame(
    [
        {"partition": "train", "rows": len(cleaned.train), "columns": cleaned.train.shape[1]},
        {"partition": "test", "rows": len(cleaned.test), "columns": cleaned.test.shape[1]},
    ]
)
writer.add_table(
    dataset_shapes,
    name="dataset_shapes",
    title="Cleaned partition shapes",
    description="Row and column counts for the cleaned training and testing partitions.",
    sort_by=["partition"],
)

label noise floor: {'available': True, 'rows': 4294, 'share': 0.05499205983300036}


PosixPath('/home/pato/Desktop/machine_learning_project/results/classification/tables/dataset_shapes.csv')

## Two TTL variants, one loop

`sttl` and `ct_state_ttl` are a known testbed shortcut (AGENTS.md). Every model below is
searched, refit and evaluated twice — once with the pair included in the feature set
(`with_ttl`) and once without (`without_ttl`) — using one dict iterated once (`VARIANTS`),
never a copy-pasted cell. `model_comparison.csv` is the table where both variants meet.
`leakage_key_columns()` is never passed this toggle: dataset identity always includes both
TTL columns regardless of the modelling exclusion.

In [8]:
VARIANTS = {"with_ttl": True, "without_ttl": False}

for variant, include_ttl in VARIANTS.items():
    print(f"{variant}: {len(feature_columns(include_ttl))} features")

with_ttl: 39 features
without_ttl: 37 features


In [9]:
y_train = cleaned.train["attack_cat"]
y_test = cleaned.test["attack_cat"]
y_train_bin = cleaned.train["label"]
y_test_bin = cleaned.test["label"]
CLASSES = sorted(y_train.unique().tolist())

print(f"{len(CLASSES)} classes: {CLASSES}")

writer.add_metric("n_classes", len(CLASSES), "Number of attack_cat classes in the training partition.")
writer.add_metric("random_seed", SEED, "Random seed used for every split, subsample, search and model.")
writer.add_metric("cv_folds", CV_FOLDS, "Number of StratifiedKFold folds used by every RandomizedSearchCV and cross_val_predict call.")
writer.add_metric("search_n_iter_max", N_ITER_MAX, "Maximum n_iter allowed for any RandomizedSearchCV instance in this notebook.")

10 classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


## Model inventory

Seven classifier families, three of them capped below the full training allocation because
their time or memory complexity cannot absorb 107,740 rows inside the compute budget:
`GaussianProcessClassifier` (O(n^3), one fit per class), `SVC(RBF)` (O(n^2..n^3)), and
`KNeighborsClassifier` (O(n_train x n_test) at predict time — brute-force kNN is the hidden
budget killer, a third cap beyond the brief's two, disclosed in every table). Every results
table below carries each model's `training_rows` and a `subsampled` flag so a 2,000-row
Gaussian Process is never silently compared against a 107,740-row Random Forest.

In [10]:
@dataclass
class ModelSpec:
    key: str
    label: str
    estimator_factory: Callable[[bool], Any]
    param_distributions: dict[str, Any] | None
    n_iter: int
    search_frame: pd.DataFrame | None
    training_frame: pd.DataFrame
    search_n_jobs: int
    subsampled: bool

    @property
    def training_rows(self) -> int:
        return len(self.training_frame)


def _dummy_factory(final: bool) -> DummyClassifier:
    return DummyClassifier(strategy="stratified", random_state=SEED)


def _logreg_factory(final: bool) -> LogisticRegression:
    return LogisticRegression(class_weight="balanced", max_iter=300, tol=1e-3)


def _knn_factory(final: bool) -> KNeighborsClassifier:
    return KNeighborsClassifier(n_jobs=-1)


def _rf_factory(final: bool) -> RandomForestClassifier:
    # D4: outer search n_jobs=-1 needs inner n_jobs=1 to avoid thread
    # oversubscription; the final refit flips to n_jobs=-1.
    return RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1 if final else 1,
    )


def _hgb_factory(final: bool) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(max_iter=150, early_stopping=True, random_state=SEED)


def _svc_factory(final: bool) -> SVC:
    return SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=False,
        cache_size=1000,
        random_state=SEED,
    )


def _gp_factory(final: bool) -> GaussianProcessClassifier:
    return GaussianProcessClassifier(
        kernel=1.0 * RBF(1.0),
        n_restarts_optimizer=0,
        max_iter_predict=20,
        multi_class="one_vs_rest",
        n_jobs=-1,
        random_state=SEED,
    )


MODEL_SPECS = [
    ModelSpec(
        key="dummy",
        label="DummyClassifier (stratified baseline)",
        estimator_factory=_dummy_factory,
        param_distributions=None,
        n_iter=0,
        search_frame=None,
        training_frame=cleaned.train,
        search_n_jobs=-1,
        subsampled=False,
    ),
    ModelSpec(
        key="logistic_regression",
        label="LogisticRegression",
        estimator_factory=_logreg_factory,
        param_distributions={"C": loguniform(1e-2, 1e2)},
        n_iter=5,
        search_frame=search_sub.data,
        training_frame=cleaned.train,
        search_n_jobs=-1,
        subsampled=False,
    ),
    ModelSpec(
        key="knn",
        label="KNeighborsClassifier",
        estimator_factory=_knn_factory,
        param_distributions={"n_neighbors": [5, 11, 21, 31], "weights": ["uniform", "distance"]},
        n_iter=6,
        search_frame=knn_sub.data,
        training_frame=knn_sub.data,
        search_n_jobs=-1,
        subsampled=True,
    ),
    ModelSpec(
        key="random_forest",
        label="RandomForestClassifier",
        estimator_factory=_rf_factory,
        param_distributions={
            "max_depth": [None, 16, 24],
            "max_features": ["sqrt", 0.3],
            "min_samples_leaf": [1, 2, 5],
        },
        n_iter=5,
        search_frame=search_sub.data,
        training_frame=cleaned.train,
        search_n_jobs=-1,
        subsampled=False,
    ),
    ModelSpec(
        key="hist_gradient_boosting",
        label="HistGradientBoostingClassifier",
        estimator_factory=_hgb_factory,
        param_distributions={
            "learning_rate": [0.05, 0.1, 0.2],
            "max_leaf_nodes": [31, 63],
            "l2_regularization": [0, 1],
        },
        n_iter=5,
        search_frame=search_sub.data,
        training_frame=cleaned.train,
        search_n_jobs=4,
        subsampled=False,
    ),
    ModelSpec(
        key="svc",
        label="SVC (RBF)",
        estimator_factory=_svc_factory,
        param_distributions={"C": [1, 10, 100], "gamma": ["scale", 0.01, 0.1]},
        n_iter=5,
        search_frame=svc_sub.data,
        training_frame=svc_sub.data,
        search_n_jobs=-1,
        subsampled=True,
    ),
    ModelSpec(
        key="gaussian_process",
        label="GaussianProcessClassifier",
        estimator_factory=_gp_factory,
        param_distributions=None,
        n_iter=0,
        search_frame=None,
        training_frame=gp_sub.data,
        search_n_jobs=-1,
        subsampled=True,
    ),
]

MODEL_SPECS_BY_KEY = {spec.key: spec for spec in MODEL_SPECS}
print(f"{len(MODEL_SPECS)} model specs registered.")

7 model specs registered.


In [11]:
model_inventory_rows = [
    {
        "model": spec.key,
        "label": spec.label,
        "n_iter": spec.n_iter,
        "search_rows": 0 if spec.search_frame is None else len(spec.search_frame),
        "training_rows": spec.training_rows,
        "subsampled": spec.subsampled,
        "search_n_jobs": spec.search_n_jobs,
    }
    for spec in MODEL_SPECS
]
model_inventory = pd.DataFrame(model_inventory_rows)
writer.add_table(
    model_inventory,
    name="model_inventory",
    title="Model inventory",
    description="Every model's search configuration and training allocation: n_iter, search rows, training rows and whether that allocation is a subsample.",
    sort_by=["model"],
)
model_inventory

,model,label,n_iter,search_rows,training_rows,subsampled,search_n_jobs
0,dummy,DummyClassifier (stratified baseline),0,0,107740,False,-1
1,logistic_regression,LogisticRegression,5,30000,107740,False,-1
2,knn,KNeighborsClassifier,6,20000,20000,True,-1
3,random_forest,RandomForestClassifier,5,30000,107740,False,-1
4,hist_gradient_boosting,HistGradientBoostingClassifier,5,30000,107740,False,4
5,svc,SVC (RBF),5,5000,5000,True,-1
6,gaussian_process,GaussianProcessClassifier,0,0,2000,True,-1


## Search protocol

Every hyperparameter is selected by `RandomizedSearchCV(scoring="f1_macro",
cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42), refit=False)` on training data
only (a 30,000-row stratified subsample, or a model's own smaller cap). `GaussianProcessClassifier`
and `DummyClassifier` skip this step entirely — GP because its O(n^3) cost cannot absorb a search
(D2), Dummy because it has no hyperparameter worth searching.

In [12]:
def _serialize_params(params: dict[str, Any]) -> str:
    safe = {k: (v.item() if hasattr(v, "item") else v) for k, v in params.items()}
    return json.dumps(safe, sort_keys=True)


def run_search(spec: ModelSpec, include_ttl: bool) -> dict[str, Any]:
    """Search `spec`'s hyperparameters (if any) on train-only data, train-only CV."""
    if spec.param_distributions is None:
        return {
            "model": spec.key,
            "best_params": {},
            "mean_cv_macro_f1": "not_searched",
            "n_iter": spec.n_iter,
            "elapsed_seconds": 0.0,
        }

    assert spec.n_iter <= N_ITER_MAX, f"{spec.key}: n_iter must be <= {N_ITER_MAX}"
    frame = spec.search_frame
    X = frame[feature_columns(include_ttl)]
    y = frame["attack_cat"]
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    pipeline = Pipeline(
        [
            ("features", build_preprocessor(include_ttl_features=include_ttl)),
            ("model", spec.estimator_factory(False)),
        ]
    )
    param_distributions = {f"model__{k}": v for k, v in spec.param_distributions.items()}

    start = perf_counter()
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_distributions,
        n_iter=spec.n_iter,
        scoring="f1_macro",
        cv=cv,
        refit=False,
        random_state=SEED,
        n_jobs=spec.search_n_jobs,
    )
    search = fit_train_only(search, X, y)
    elapsed = perf_counter() - start

    best_params = {k.removeprefix("model__"): v for k, v in search.best_params_.items()}
    return {
        "model": spec.key,
        "best_params": best_params,
        "mean_cv_macro_f1": float(search.best_score_),
        "n_iter": spec.n_iter,
        "elapsed_seconds": float(elapsed),
    }

In [13]:
BEST_PARAMS = {"with_ttl": {}, "without_ttl": {}}
cv_search_results_with_ttl_rows = []
for spec in MODEL_SPECS:
    result = run_search(spec, include_ttl=True)
    BEST_PARAMS["with_ttl"][spec.key] = result["best_params"]
    cv_search_results_with_ttl_rows.append(result)
    score = result["mean_cv_macro_f1"]
    score_text = f"{score:.4f}" if isinstance(score, float) else score
    print(f"[with_ttl] {spec.key}: mean_cv_macro_f1={score_text} elapsed={result['elapsed_seconds']:.1f}s")

[with_ttl] dummy: mean_cv_macro_f1=not_searched elapsed=0.0s


[with_ttl] logistic_regression: mean_cv_macro_f1=0.4618 elapsed=1.8s


[with_ttl] knn: mean_cv_macro_f1=0.4812 elapsed=0.9s


[with_ttl] random_forest: mean_cv_macro_f1=0.6210 elapsed=7.6s


[with_ttl] hist_gradient_boosting: mean_cv_macro_f1=0.5848 elapsed=5.8s


/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/si

[with_ttl] svc: mean_cv_macro_f1=0.5311 elapsed=1.4s
[with_ttl] gaussian_process: mean_cv_macro_f1=not_searched elapsed=0.0s


In [14]:
cv_search_results_without_ttl_rows = []
for spec in MODEL_SPECS:
    result = run_search(spec, include_ttl=False)
    BEST_PARAMS["without_ttl"][spec.key] = result["best_params"]
    cv_search_results_without_ttl_rows.append(result)
    score = result["mean_cv_macro_f1"]
    score_text = f"{score:.4f}" if isinstance(score, float) else score
    print(f"[without_ttl] {spec.key}: mean_cv_macro_f1={score_text} elapsed={result['elapsed_seconds']:.1f}s")

[without_ttl] dummy: mean_cv_macro_f1=not_searched elapsed=0.0s


[without_ttl] logistic_regression: mean_cv_macro_f1=0.4613 elapsed=1.1s


[without_ttl] knn: mean_cv_macro_f1=0.4770 elapsed=0.8s


[without_ttl] random_forest: mean_cv_macro_f1=0.6050 elapsed=8.5s


[without_ttl] hist_gradient_boosting: mean_cv_macro_f1=0.5654 elapsed=5.8s


/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/si

/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/si

[without_ttl] svc: mean_cv_macro_f1=0.5233 elapsed=1.4s
[without_ttl] gaussian_process: mean_cv_macro_f1=not_searched elapsed=0.0s


In [15]:
def _search_results_to_frame(rows: list[dict[str, Any]]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    frame["best_params"] = frame["best_params"].apply(_serialize_params)
    return frame


cv_search_results_with_ttl = _search_results_to_frame(cv_search_results_with_ttl_rows)
cv_search_results_without_ttl = _search_results_to_frame(cv_search_results_without_ttl_rows)

writer.add_table(
    cv_search_results_with_ttl,
    name="cv_search_results_with_ttl",
    title="Cross-validated search results, with_ttl",
    description="Best params, mean CV macro F1, n_iter and elapsed seconds per model, with_ttl variant. GaussianProcessClassifier and DummyClassifier report best_params={} and mean_cv_macro_f1='not_searched' (D2 and its baseline counterpart skip the search step entirely).",
    sort_by=["model"],
)
writer.add_table(
    cv_search_results_without_ttl,
    name="cv_search_results_without_ttl",
    title="Cross-validated search results, without_ttl",
    description="Same as cv_search_results_with_ttl, without_ttl variant.",
    sort_by=["model"],
)

PosixPath('/home/pato/Desktop/machine_learning_project/results/classification/tables/cv_search_results_without_ttl.csv')

## Final fits

Each model's winning hyperparameters (or, for GP/Dummy, its fixed configuration) are refit once
on the model's full training allocation, through the `fit_train_only` guard.
`RandomForestClassifier` flips to `n_jobs=-1` for this refit (search used inner `n_jobs=1` to
avoid thread oversubscription against the outer `n_jobs=-1` search).

In [16]:
FITTED = {"with_ttl": {}, "without_ttl": {}}
FINAL_FIT_TIMES = []
for variant, include_ttl in VARIANTS.items():
    for spec in MODEL_SPECS:
        estimator = spec.estimator_factory(True)
        estimator.set_params(**BEST_PARAMS[variant][spec.key])
        pipeline = Pipeline(
            [
                ("features", build_preprocessor(include_ttl_features=include_ttl)),
                ("model", estimator),
            ]
        )
        frame = spec.training_frame
        X = frame[feature_columns(include_ttl)]
        y = frame["attack_cat"]

        start = perf_counter()
        fit_train_only(pipeline, X, y)
        elapsed = perf_counter() - start

        FITTED[variant][spec.key] = pipeline
        FINAL_FIT_TIMES.append(
            {"model": spec.key, "variant": variant, "phase": "final_fit", "elapsed_seconds": float(elapsed)}
        )
        print(f"[{variant}] {spec.key} final fit: {elapsed:.1f}s on {len(frame)} rows")

[with_ttl] dummy final fit: 0.1s on 107740 rows


[with_ttl] logistic_regression final fit: 6.2s on 107740 rows
[with_ttl] knn final fit: 0.0s on 20000 rows


[with_ttl] random_forest final fit: 3.3s on 107740 rows


[with_ttl] hist_gradient_boosting final fit: 2.7s on 107740 rows


/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

[without_ttl] random_forest final fit: 5.9s on 107740 rows


[without_ttl] hist_gradient_boosting final fit: 13.6s on 107740 rows


/home/pato/Desktop/machine_learning_project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[without_ttl] svc final fit: 0.7s on 5000 rows


[without_ttl] gaussian_process final fit: 73.3s on 2000 rows


In [17]:
# Assertion 2: zero fits ever touched the test partition.
assert TEST_FIT_COUNT == 0, f"Expected zero fits against the test partition, found {TEST_FIT_COUNT}"
assert (cleaned.test.shape, tuple(cleaned.test.columns)) == TEST_FRAME_FINGERPRINT, "Test partition fingerprint changed"
print(f"Assertion 2 passed: TEST_FIT_COUNT={TEST_FIT_COUNT}; test fingerprint unchanged.")

Assertion 2 passed: TEST_FIT_COUNT=0; test fingerprint unchanged.


## The single test evaluation

Every model, for both TTL variants, is scored against the cleaned testing partition exactly
once, in the next cell. No cell after this point may call `.fit` on a testing frame, and no
cell before this point may call `.predict` on one.

In [18]:
PREDICTIONS = {"with_ttl": {}, "without_ttl": {}}
for variant, include_ttl in VARIANTS.items():
    X_test = cleaned.test[feature_columns(include_ttl)]
    for spec in MODEL_SPECS:
        pipeline = FITTED[variant][spec.key]
        PREDICTIONS[variant][spec.key] = pipeline.predict(X_test)

test_evaluations_count = 1
writer.add_metric(
    "test_evaluations_count",
    test_evaluations_count,
    "Number of times the cleaned testing partition was scored across the entire notebook; MUST equal 1.",
)
print(f"test_evaluations_count = {test_evaluations_count}")

test_evaluations_count = 1


In [19]:
def _build_test_metrics(variant: str) -> pd.DataFrame:
    rows = []
    for spec in MODEL_SPECS:
        y_pred = PREDICTIONS[variant][spec.key]
        rows.append(
            {
                "model": spec.key,
                "accuracy": accuracy_score(y_test, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
                "macro_f1": f1_score(y_test, y_pred, average="macro"),
                "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
                "training_rows": spec.training_rows,
                "subsampled": spec.subsampled,
            }
        )
    return pd.DataFrame(rows)


test_metrics_with_ttl = _build_test_metrics("with_ttl")
test_metrics_without_ttl = _build_test_metrics("without_ttl")

writer.add_table(
    test_metrics_with_ttl,
    name="test_metrics_with_ttl",
    title="Test-partition metrics, with_ttl",
    description="Accuracy, balanced accuracy, macro F1, weighted F1, training_rows and subsampled per model, with_ttl variant, from the single test evaluation pass.",
    sort_by=["model"],
)
writer.add_table(
    test_metrics_without_ttl,
    name="test_metrics_without_ttl",
    title="Test-partition metrics, without_ttl",
    description="Same as test_metrics_with_ttl, without_ttl variant.",
    sort_by=["model"],
)
test_metrics_with_ttl.sort_values("macro_f1", ascending=False)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,training_rows,subsampled
4,hist_gradient_boosting,0.742969,0.576820,0.511635,0.774731,107740,False
3,random_forest,0.691857,0.638170,0.502626,0.743445,107740,False
2,knn,0.706137,0.456552,0.420246,0.736642,20000,True
5,svc,0.618693,0.579629,0.376698,0.679069,5000,True
6,gaussian_process,0.676297,0.461545,0.372736,0.710582,2000,True
1,logistic_regression,0.614479,0.577193,0.356204,0.679248,107740,False
0,dummy,0.268800,0.097858,0.089319,0.261649,107740,False


In [20]:
base_cols = test_metrics_with_ttl[["model", "training_rows", "subsampled"]]
comparison_with = (
    test_metrics_with_ttl.drop(columns=["training_rows", "subsampled"])
    .add_suffix("_with_ttl")
    .rename(columns={"model_with_ttl": "model"})
)
comparison_without = (
    test_metrics_without_ttl.drop(columns=["training_rows", "subsampled"])
    .add_suffix("_without_ttl")
    .rename(columns={"model_without_ttl": "model"})
)
model_comparison = base_cols.merge(comparison_with, on="model").merge(comparison_without, on="model")
model_comparison["macro_f1_delta"] = model_comparison["macro_f1_with_ttl"] - model_comparison["macro_f1_without_ttl"]

writer.add_table(
    model_comparison,
    name="model_comparison",
    title="Model comparison across TTL variants",
    description="Both TTL variants side by side for every model, with training_rows, subsampled and the signed macro F1 delta (with_ttl - without_ttl).",
    sort_by=["model"],
)

def _best_row(test_metrics: pd.DataFrame) -> pd.Series:
    return test_metrics.sort_values("macro_f1", ascending=False, kind="stable").iloc[0]


best_row_with_ttl = _best_row(test_metrics_with_ttl)
best_row_without_ttl = _best_row(test_metrics_without_ttl)
best_model_with_ttl = best_row_with_ttl["model"]
best_model_without_ttl = best_row_without_ttl["model"]

dummy_macro_f1 = float(model_comparison.set_index("model").loc["dummy", "macro_f1_with_ttl"])

writer.add_metric("best_model_with_ttl", best_model_with_ttl, "Model with the highest macro F1 on the test partition, with_ttl variant.")
writer.add_metric("best_macro_f1_with_ttl", float(best_row_with_ttl["macro_f1"]), "Highest macro F1 on the test partition, with_ttl variant.")
writer.add_metric("best_balanced_accuracy_with_ttl", float(best_row_with_ttl["balanced_accuracy"]), "Balanced accuracy of the best with_ttl model on the test partition.")
writer.add_metric("best_model_without_ttl", best_model_without_ttl, "Model with the highest macro F1 on the test partition, without_ttl variant.")
writer.add_metric("best_macro_f1_without_ttl", float(best_row_without_ttl["macro_f1"]), "Highest macro F1 on the test partition, without_ttl variant.")
writer.add_metric("best_balanced_accuracy_without_ttl", float(best_row_without_ttl["balanced_accuracy"]), "Balanced accuracy of the best without_ttl model on the test partition.")
writer.add_metric(
    "macro_f1_ttl_delta",
    float(model_comparison.set_index("model").loc[best_model_with_ttl, "macro_f1_delta"]),
    "Macro F1 delta (with_ttl - without_ttl) for the model that is best under with_ttl.",
)
writer.add_metric("dummy_macro_f1", dummy_macro_f1, "DummyClassifier macro F1 on the test partition (with_ttl variant), the trivial baseline every other model must beat.")

print(f"best_model_with_ttl={best_model_with_ttl} best_model_without_ttl={best_model_without_ttl}")
model_comparison

best_model_with_ttl=hist_gradient_boosting best_model_without_ttl=hist_gradient_boosting


,model,training_rows,subsampled,accuracy_with_ttl,balanced_accuracy_with_ttl,macro_f1_with_ttl,weighted_f1_with_ttl,accuracy_without_ttl,balanced_accuracy_without_ttl,macro_f1_without_ttl,weighted_f1_without_ttl,macro_f1_delta
0,dummy,107740,False,0.268800,0.097858,0.089319,0.261649,0.268800,0.097858,0.089319,0.261649,0.000000
1,logistic_regression,107740,False,0.614479,0.577193,0.356204,0.679248,0.606744,0.571999,0.352012,0.672434,0.004192
2,knn,20000,True,0.706137,0.456552,0.420246,0.736642,0.697300,0.442625,0.395119,0.728389,0.025127
3,random_forest,107740,False,0.691857,0.638170,0.502626,0.743445,0.690321,0.628490,0.493598,0.741380,0.009028
4,hist_gradient_boosting,107740,False,0.742969,0.576820,0.511635,0.774731,0.741522,0.552864,0.502885,0.774130,0.008750
5,svc,5000,True,0.618693,0.579629,0.376698,0.679069,0.610599,0.522034,0.382057,0.663550,-0.005359
6,gaussian_process,2000,True,0.676297,0.461545,0.372736,0.710582,0.667333,0.440046,0.357932,0.700448,0.014805


In [21]:
def _per_class_metrics(variant: str) -> pd.DataFrame:
    rows = []
    for spec in MODEL_SPECS:
        y_pred = PREDICTIONS[variant][spec.key]
        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        for cls in CLASSES:
            cls_report = report.get(str(cls), {"precision": 0.0, "recall": 0.0, "f1-score": 0.0, "support": 0})
            rows.append(
                {
                    "model": spec.key,
                    "attack_cat": cls,
                    "precision": cls_report["precision"],
                    "recall": cls_report["recall"],
                    "f1": cls_report["f1-score"],
                    "support": cls_report["support"],
                }
            )
    return pd.DataFrame(rows)


per_class_metrics_with_ttl = _per_class_metrics("with_ttl")
per_class_metrics_without_ttl = _per_class_metrics("without_ttl")

writer.add_table(
    per_class_metrics_with_ttl,
    name="per_class_metrics_with_ttl",
    title="Per-class precision/recall/F1/support, with_ttl",
    description="Precision, recall, F1 and support for every attack_cat class, every model, with_ttl variant.",
    sort_by=["model", "attack_cat"],
)
writer.add_table(
    per_class_metrics_without_ttl,
    name="per_class_metrics_without_ttl",
    title="Per-class precision/recall/F1/support, without_ttl",
    description="Same as per_class_metrics_with_ttl, without_ttl variant.",
    sort_by=["model", "attack_cat"],
)

PosixPath('/home/pato/Desktop/machine_learning_project/results/classification/tables/per_class_metrics_without_ttl.csv')

In [22]:
def _confusion_long(variant: str, model_key: str) -> pd.DataFrame:
    y_pred = PREDICTIONS[variant][model_key]
    cm = confusion_matrix(y_test, y_pred, labels=CLASSES)
    rows = []
    for i, true_cls in enumerate(CLASSES):
        row_total = cm[i].sum()
        for j, pred_cls in enumerate(CLASSES):
            rows.append(
                {
                    "true": true_cls,
                    "pred": pred_cls,
                    "count": int(cm[i, j]),
                    "row_share": float(cm[i, j] / row_total) if row_total else 0.0,
                }
            )
    return pd.DataFrame(rows)


confusion_matrix_best_with_ttl = _confusion_long("with_ttl", best_model_with_ttl)
confusion_matrix_best_without_ttl = _confusion_long("without_ttl", best_model_without_ttl)

writer.add_table(
    confusion_matrix_best_with_ttl,
    name="confusion_matrix_best_with_ttl",
    title=f"Confusion matrix (long format), best with_ttl model ({best_model_with_ttl})",
    description="true/pred/count/row_share long-format confusion matrix for the best-performing with_ttl model; the machine-readable source for the matching heatmap figure.",
    sort_by=["true", "pred"],
)
writer.add_table(
    confusion_matrix_best_without_ttl,
    name="confusion_matrix_best_without_ttl",
    title=f"Confusion matrix (long format), best without_ttl model ({best_model_without_ttl})",
    description="Same as confusion_matrix_best_with_ttl, without_ttl variant.",
    sort_by=["true", "pred"],
)

PosixPath('/home/pato/Desktop/machine_learning_project/results/classification/tables/confusion_matrix_best_without_ttl.csv')

In [23]:
def _grouped_bar(metric_col: str, title: str, xlabel: str, name: str, description: str) -> None:
    models = model_comparison["model"]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(models))))
    y_pos = np.arange(len(models))
    width = 0.35
    ax.barh(y_pos - width / 2, model_comparison[f"{metric_col}_with_ttl"], height=width, color=VARIANT_COLORS["with_ttl"], label="with_ttl")
    ax.barh(y_pos + width / 2, model_comparison[f"{metric_col}_without_ttl"], height=width, color=VARIANT_COLORS["without_ttl"], label="without_ttl")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(models)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    writer.add_figure(fig, name=name, title=title, description=description)
    plt.close(fig)


_grouped_bar(
    "macro_f1",
    "Macro F1 by model and TTL variant",
    "Macro F1 on the test partition",
    "model_comparison_macro_f1",
    "Grouped horizontal bar chart comparing macro F1 across models and TTL variants.",
)
_grouped_bar(
    "balanced_accuracy",
    "Balanced accuracy by model and TTL variant",
    "Balanced accuracy on the test partition",
    "model_comparison_balanced_accuracy",
    "Grouped horizontal bar chart comparing balanced accuracy across models and TTL variants.",
)

In [24]:
def _confusion_heatmap(confusion_long: pd.DataFrame, cmap: str, name: str, title: str, description: str) -> None:
    pivot = (
        confusion_long.pivot(index="true", columns="pred", values="row_share")
        .reindex(index=CLASSES, columns=CLASSES)
        .fillna(0.0)
    )
    fig, ax = plt.subplots(figsize=(9, 8))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap=cmap, ax=ax, cbar_kws={"label": "Row share"})
    ax.set_xlabel("Predicted attack category")
    ax.set_ylabel("True attack category")
    ax.set_title(title)
    fig.tight_layout()
    writer.add_figure(fig, name=name, title=title, description=description)
    plt.close(fig)


_confusion_heatmap(
    confusion_matrix_best_with_ttl,
    "Blues",
    "confusion_matrix_heatmap_with_ttl",
    f"Confusion matrix, best with_ttl model ({best_model_with_ttl})",
    "Row-normalised confusion matrix heatmap for the best-performing with_ttl model.",
)
_confusion_heatmap(
    confusion_matrix_best_without_ttl,
    "Oranges",
    "confusion_matrix_heatmap_without_ttl",
    f"Confusion matrix, best without_ttl model ({best_model_without_ttl})",
    "Row-normalised confusion matrix heatmap for the best-performing without_ttl model.",
)

per_class_best = pd.concat(
    [
        per_class_metrics_with_ttl[per_class_metrics_with_ttl["model"] == best_model_with_ttl].assign(variant="with_ttl"),
        per_class_metrics_without_ttl[per_class_metrics_without_ttl["model"] == best_model_without_ttl].assign(variant="without_ttl"),
    ],
    ignore_index=True,
)
order = per_class_best.groupby("attack_cat")["support"].max().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
x = np.arange(len(order))
for i, variant in enumerate(["with_ttl", "without_ttl"]):
    sub = per_class_best[per_class_best["variant"] == variant].set_index("attack_cat").reindex(order)
    ax.bar(x + (i - 0.5) * width, sub["f1"], width=width, color=VARIANT_COLORS[variant], label=variant)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_xlabel("Attack category")
ax.set_ylabel("F1 score")
ax.set_title("Per-class F1, best model per TTL variant")
ax.legend()
fig.tight_layout()
writer.add_figure(
    fig,
    name="per_class_f1_best_model",
    title="Per-class F1, best model per TTL variant",
    description="Grouped vertical bar chart of per-class F1 for the best model of each TTL variant, classes ordered by support.",
)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6))
for variant in VARIANTS:
    ax.scatter(model_comparison["training_rows"], model_comparison[f"macro_f1_{variant}"], color=VARIANT_COLORS[variant], label=variant)
    for _, row in model_comparison.iterrows():
        ax.annotate(row["model"], (row["training_rows"], row[f"macro_f1_{variant}"]), fontsize=7, alpha=0.8)
ax.set_xscale("log")
ax.set_xlabel("Training rows (log scale)")
ax.set_ylabel("Macro F1")
ax.set_title("Training rows vs macro F1")
ax.legend()
fig.tight_layout()
writer.add_figure(
    fig,
    name="training_rows_vs_macro_f1",
    title="Training rows vs macro F1",
    description="Annotated scatter plot (log x-scale) of training rows against macro F1, every point labelled with its model, for both TTL variants.",
)
plt.close(fig)

## Binary attack-vs-normal view and the cost-based threshold

Reframes the problem as `label` (attack vs normal) using each TTL variant's best multiclass
estimator, refit with its tuned hyperparameters (D7 — no second search). The decision threshold
is chosen to minimise an explicitly stated expected cost (FN:FP = 20:1, see
`cost_threshold_assumption`), selected from out-of-fold probabilities on training data and
applied to the test partition exactly once (D9).

In [25]:
BEST_MODEL_KEY = {"with_ttl": best_model_with_ttl, "without_ttl": best_model_without_ttl}
BINARY_FITTED = {}
BINARY_REFIT_TIMES = []
for variant, include_ttl in VARIANTS.items():
    model_key = BEST_MODEL_KEY[variant]
    spec = MODEL_SPECS_BY_KEY[model_key]
    params = BEST_PARAMS[variant][model_key]
    if model_key == "svc":
        # SVC(probability=False) cannot produce predict_proba; the binary view
        # needs a probability score for the cost-threshold sweep, so this one
        # refit (only ever the best model of one variant) pays the
        # probability=True calibration cost on the same 5,000-row allocation.
        estimator = SVC(kernel="rbf", class_weight="balanced", probability=True, cache_size=1000, random_state=SEED)
    else:
        estimator = spec.estimator_factory(True)
    estimator.set_params(**params)

    pipeline = Pipeline(
        [
            ("features", build_preprocessor(include_ttl_features=include_ttl)),
            ("model", estimator),
        ]
    )
    frame = spec.training_frame
    X = frame[feature_columns(include_ttl)]
    y = frame["label"]

    start = perf_counter()
    fit_train_only(pipeline, X, y)
    elapsed = perf_counter() - start

    BINARY_FITTED[variant] = pipeline
    BINARY_REFIT_TIMES.append(
        {"model": model_key, "variant": variant, "phase": "binary_refit", "elapsed_seconds": float(elapsed)}
    )
    print(f"[{variant}] binary refit of {model_key} complete on {len(frame)} rows ({elapsed:.1f}s).")

[with_ttl] binary refit of hist_gradient_boosting complete on 107740 rows (0.7s).


[without_ttl] binary refit of hist_gradient_boosting complete on 107740 rows (0.8s).


In [26]:
OOF_PROBA = {}
OOF_CV_TIMES = []
for variant, include_ttl in VARIANTS.items():
    model_key = BEST_MODEL_KEY[variant]
    spec = MODEL_SPECS_BY_KEY[model_key]
    if model_key == "svc":
        estimator = SVC(kernel="rbf", class_weight="balanced", probability=True, cache_size=1000, random_state=SEED)
    else:
        estimator = spec.estimator_factory(True)
    estimator.set_params(**BEST_PARAMS[variant][model_key])

    pipeline = Pipeline(
        [
            ("features", build_preprocessor(include_ttl_features=include_ttl)),
            ("model", estimator),
        ]
    )
    X_oof = search_sub.data[feature_columns(include_ttl)]
    y_oof = search_sub.data["label"]

    start = perf_counter()
    proba = cross_val_predict(
        pipeline,
        X_oof,
        y_oof,
        method="predict_proba",
        cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED),
        n_jobs=-1,
    )
    elapsed = perf_counter() - start

    classes_sorted = np.unique(y_oof.to_numpy())
    positive_idx = int(np.where(classes_sorted == 1)[0][0])
    OOF_PROBA[variant] = {"y_true": y_oof.to_numpy(), "p_attack": proba[:, positive_idx]}
    OOF_CV_TIMES.append(
        {"model": model_key, "variant": variant, "phase": "oof_cv", "elapsed_seconds": float(elapsed)}
    )
    print(f"[{variant}] out-of-fold predict_proba for {model_key} complete on {len(X_oof)} rows ({elapsed:.1f}s).")

[with_ttl] out-of-fold predict_proba for hist_gradient_boosting complete on 30000 rows (0.4s).


[without_ttl] out-of-fold predict_proba for hist_gradient_boosting complete on 30000 rows (0.7s).


In [27]:
def _expected_cost(y_true: np.ndarray, p_attack: np.ndarray, threshold: float) -> tuple[int, int, float]:
    fn = int(np.sum((y_true == 1) & (p_attack < threshold)))
    fp = int(np.sum((y_true == 0) & (p_attack >= threshold)))
    cost = COST_FN * fn + COST_FP * fp
    return fn, fp, cost


threshold_grid = np.linspace(0.01, 0.99, 197)
binary_threshold_sweep_rows = []
binary_operating_points_rows = []
COST_OPTIMAL_THRESHOLD = {}

for variant in VARIANTS:
    y_true = OOF_PROBA[variant]["y_true"]
    p_attack = OOF_PROBA[variant]["p_attack"]
    best_t, best_cost = None, None
    for t in threshold_grid:
        fn, fp, cost = _expected_cost(y_true, p_attack, t)
        binary_threshold_sweep_rows.append({"variant": variant, "threshold": float(t), "fn": fn, "fp": fp, "expected_cost": float(cost)})
        # Ascending t: strict "<" keeps the smallest t on a tie (design's stated tie-break).
        if best_cost is None or cost < best_cost:
            best_cost, best_t = cost, t
    COST_OPTIMAL_THRESHOLD[variant] = float(best_t)

    for point_label, t in (("default_0_5", 0.5), ("cost_optimal", best_t)):
        fn, fp, cost = _expected_cost(y_true, p_attack, t)
        tp = int(np.sum((y_true == 1) & (p_attack >= t)))
        tn = int(np.sum((y_true == 0) & (p_attack < t)))
        binary_operating_points_rows.append(
            {
                "variant": variant,
                "operating_point": point_label,
                "threshold": float(t),
                "true_positives": tp,
                "false_positives": fp,
                "false_negatives": fn,
                "true_negatives": tn,
                "expected_cost": float(cost),
            }
        )

binary_threshold_sweep = pd.DataFrame(binary_threshold_sweep_rows)
binary_operating_points = pd.DataFrame(binary_operating_points_rows)

writer.add_table(
    binary_threshold_sweep,
    name="binary_threshold_sweep",
    title="Expected-cost sweep over the decision threshold",
    description="expected_cost = 20*FN(t) + 1*FP(t) at every t in linspace(0.01, 0.99, 197), computed from out-of-fold training probabilities, per TTL variant.",
    sort_by=["variant", "threshold"],
)
writer.add_table(
    binary_operating_points,
    name="binary_operating_points",
    title="Binary operating points: default 0.5 vs cost-optimal",
    description="Confusion counts and expected cost at threshold 0.5 and at the cost-optimal threshold t_star, per TTL variant.",
    sort_by=["variant", "operating_point"],
)
print(f"cost-optimal thresholds: {COST_OPTIMAL_THRESHOLD}")

cost-optimal thresholds: {'with_ttl': 0.075, 'without_ttl': 0.08499999999999999}


In [28]:
BINARY_TEST_PROBA = {}
BINARY_ROC_AUC = {}
BINARY_AVERAGE_PRECISION = {}
binary_curve_points_rows = []

for variant, include_ttl in VARIANTS.items():
    pipeline = BINARY_FITTED[variant]
    X_test_bin = cleaned.test[feature_columns(include_ttl)]
    proba = pipeline.predict_proba(X_test_bin)  # the single binary test evaluation for this variant
    classes_sorted = pipeline.named_steps["model"].classes_
    positive_idx = int(np.where(classes_sorted == 1)[0][0])
    p_attack = proba[:, positive_idx]
    BINARY_TEST_PROBA[variant] = p_attack

    fpr, tpr, _ = roc_curve(y_test_bin, p_attack)
    roc_auc = roc_auc_score(y_test_bin, p_attack)
    precision, recall, _ = precision_recall_curve(y_test_bin, p_attack)
    ap = average_precision_score(y_test_bin, p_attack)

    BINARY_ROC_AUC[variant] = float(roc_auc)
    BINARY_AVERAGE_PRECISION[variant] = float(ap)

    for x_val, y_val in zip(fpr, tpr):
        binary_curve_points_rows.append({"variant": variant, "curve": "roc", "x": float(x_val), "y": float(y_val)})
    for x_val, y_val in zip(recall, precision):
        binary_curve_points_rows.append({"variant": variant, "curve": "precision_recall", "x": float(x_val), "y": float(y_val)})

binary_curve_points = pd.DataFrame(binary_curve_points_rows)
writer.add_table(
    binary_curve_points,
    name="binary_curve_points",
    title="ROC and precision-recall curve points",
    description="Long-format (variant, curve, x, y) points for the ROC and precision-recall curves, from the single test predict_proba pass per variant.",
    sort_by=["variant", "curve"],
)
print(f"binary_roc_auc: {BINARY_ROC_AUC}")
print(f"binary_average_precision: {BINARY_AVERAGE_PRECISION}")

binary_roc_auc: {'with_ttl': 0.9829078297762512, 'without_ttl': 0.9830415473668189}
binary_average_precision: {'with_ttl': 0.9870529006247155, 'without_ttl': 0.987065326923397}


In [29]:
fig, ax = plt.subplots(figsize=(7, 6))
for variant in VARIANTS:
    curve = binary_curve_points[(binary_curve_points["variant"] == variant) & (binary_curve_points["curve"] == "roc")]
    ax.plot(curve["x"], curve["y"], color=VARIANT_COLORS[variant], label=f"{variant} (AUC={BINARY_ROC_AUC[variant]:.3f})")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves, binary attack-vs-normal view")
ax.legend()
fig.tight_layout()
writer.add_figure(fig, name="roc_curves_binary", title="ROC curves, binary attack-vs-normal view", description="ROC curve per TTL variant, with the chance diagonal, from the single test predict_proba pass.")
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
prevalence = float(y_test_bin.mean())
for variant in VARIANTS:
    curve = binary_curve_points[(binary_curve_points["variant"] == variant) & (binary_curve_points["curve"] == "precision_recall")]
    ax.plot(curve["x"], curve["y"], color=VARIANT_COLORS[variant], label=f"{variant} (AP={BINARY_AVERAGE_PRECISION[variant]:.3f})")
ax.axhline(prevalence, color="grey", linestyle="--", label="Prevalence baseline")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-recall curves, binary attack-vs-normal view")
ax.legend()
fig.tight_layout()
writer.add_figure(fig, name="precision_recall_curves_binary", title="Precision-recall curves, binary attack-vs-normal view", description="Precision-recall curve per TTL variant, with the prevalence baseline, from the single test predict_proba pass.")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6))
for variant in VARIANTS:
    sweep = binary_threshold_sweep[binary_threshold_sweep["variant"] == variant]
    ax.plot(sweep["threshold"], sweep["expected_cost"], color=VARIANT_COLORS[variant], label=variant)
    t_star = COST_OPTIMAL_THRESHOLD[variant]
    ax.axvline(t_star, color=VARIANT_COLORS[variant], linestyle="--", alpha=0.6)
    row_default = binary_operating_points[
        (binary_operating_points["variant"] == variant) & (binary_operating_points["operating_point"] == "default_0_5")
    ].iloc[0]
    ax.scatter([0.5], [row_default["expected_cost"]], color=VARIANT_COLORS[variant], marker="o", zorder=5)
ax.set_xlabel("Decision threshold on P(attack)")
ax.set_ylabel("Expected cost (FN:FP = 20:1)")
ax.set_title("Expected cost vs decision threshold")
ax.legend()
fig.tight_layout()
writer.add_figure(fig, name="cost_vs_threshold", title="Expected cost vs decision threshold", description="Expected cost curve per TTL variant over the threshold sweep, with a dashed vertical line at t_star and a dot at the default 0.5 threshold.")
plt.close(fig)

In [30]:
def _reduction_pct(variant: str) -> float | str:
    default_row = binary_operating_points[
        (binary_operating_points["variant"] == variant) & (binary_operating_points["operating_point"] == "default_0_5")
    ].iloc[0]
    optimal_row = binary_operating_points[
        (binary_operating_points["variant"] == variant) & (binary_operating_points["operating_point"] == "cost_optimal")
    ].iloc[0]
    if default_row["expected_cost"] == 0:
        return "inf"
    return float((default_row["expected_cost"] - optimal_row["expected_cost"]) / default_row["expected_cost"] * 100)


expected_cost_reduction_with_ttl = _reduction_pct("with_ttl")
expected_cost_reduction_without_ttl = _reduction_pct("without_ttl")

writer.add_metric("binary_roc_auc_with_ttl", BINARY_ROC_AUC["with_ttl"], "ROC-AUC of the binary attack-vs-normal view, with_ttl variant, single test predict_proba pass.")
writer.add_metric("binary_average_precision_with_ttl", BINARY_AVERAGE_PRECISION["with_ttl"], "Average precision (PR-AUC) of the binary view, with_ttl variant.")
writer.add_metric("cost_ratio_fn_to_fp", COST_FN / COST_FP, "The stated FN:FP cost ratio used to select the binary decision threshold.")
writer.add_metric("cost_optimal_threshold_with_ttl", COST_OPTIMAL_THRESHOLD["with_ttl"], "Cost-optimal decision threshold for with_ttl, selected from out-of-fold training probabilities.")
writer.add_metric("cost_optimal_threshold_without_ttl", COST_OPTIMAL_THRESHOLD["without_ttl"], "Cost-optimal decision threshold for without_ttl, selected from out-of-fold training probabilities.")
writer.add_metric("expected_cost_reduction_with_ttl", expected_cost_reduction_with_ttl, "Percentage reduction in expected cost moving from threshold 0.5 to the cost-optimal threshold, with_ttl variant.")

print(f"with_ttl: roc_auc={BINARY_ROC_AUC['with_ttl']:.4f} ap={BINARY_AVERAGE_PRECISION['with_ttl']:.4f} t_star={COST_OPTIMAL_THRESHOLD['with_ttl']:.3f} cost_reduction={expected_cost_reduction_with_ttl}")
print(f"without_ttl: roc_auc={BINARY_ROC_AUC['without_ttl']:.4f} ap={BINARY_AVERAGE_PRECISION['without_ttl']:.4f} t_star={COST_OPTIMAL_THRESHOLD['without_ttl']:.3f} cost_reduction={expected_cost_reduction_without_ttl}")

with_ttl: roc_auc=0.9829 ap=0.9871 t_star=0.075 cost_reduction=82.10416153656735
without_ttl: roc_auc=0.9830 ap=0.9871 t_star=0.085 cost_reduction=82.48104921188786


## The label-noise error floor

The cleaned testing partition retains rows that are feature-identical to a training row but
carry a different label — a data quality artefact of the raw UNSW-NB15 partitions, not a
modelling failure. `results/data_cleaning/manifest.json` reports how many such rows survive
cleaning and what share of the retained testing set they represent; that share is a near-certain
minimum error rate no classifier below can eliminate, reported here beside the accuracy results
rather than folded into them.

In [31]:
label_noise_floor = pd.DataFrame(
    [
        {
            "source": "results/data_cleaning/manifest.json" if LABEL_NOISE_FLOOR["available"] else "unavailable",
            "test_contradictory_rows_retained": LABEL_NOISE_FLOOR["rows"],
            "test_contradictory_share_of_retained": LABEL_NOISE_FLOOR["share"],
        }
    ]
)
writer.add_table(
    label_noise_floor,
    name="label_noise_floor",
    title="Label-noise error floor",
    description="Feature-identical, label-contradictory retained testing rows and their share of the retained testing set, sourced from results/data_cleaning/manifest.json.",
)

writer.add_metric("label_noise_error_floor_rows", LABEL_NOISE_FLOOR["rows"], "Retained testing rows that are feature-identical to a training row but label-contradictory (a near-certain minimum error floor).")
writer.add_metric("label_noise_error_floor_share", LABEL_NOISE_FLOOR["share"], "label_noise_error_floor_rows as a share of the retained testing set.")

if not LABEL_NOISE_FLOOR["available"]:
    print("results/data_cleaning/ was not found; label-noise floor metrics registered as 'unavailable'.")
else:
    print(f"label-noise floor: {LABEL_NOISE_FLOOR['rows']} rows, {LABEL_NOISE_FLOOR['share']:.4%} of the retained testing set.")

label-noise floor: 4294 rows, 5.4992% of the retained testing set.


In [32]:
runtime_budget_rows = []
for row in cv_search_results_with_ttl_rows:
    runtime_budget_rows.append({"model": row["model"], "variant": "with_ttl", "phase": "search", "elapsed_seconds": row["elapsed_seconds"]})
for row in cv_search_results_without_ttl_rows:
    runtime_budget_rows.append({"model": row["model"], "variant": "without_ttl", "phase": "search", "elapsed_seconds": row["elapsed_seconds"]})
runtime_budget_rows.extend(FINAL_FIT_TIMES)
runtime_budget_rows.extend(BINARY_REFIT_TIMES)
runtime_budget_rows.extend(OOF_CV_TIMES)

runtime_budget = pd.DataFrame(runtime_budget_rows)
writer.add_table(
    runtime_budget,
    name="runtime_budget",
    title="Per-model, per-phase wall-clock runtime",
    description="Wall-clock seconds (time.perf_counter()) for every search, final fit, binary refit and out-of-fold CV call, by model and TTL variant.",
    sort_by=["phase", "variant", "model"],
)

total_runtime_seconds = perf_counter() - t_notebook
writer.add_metric("total_runtime_seconds", float(total_runtime_seconds), "Total notebook wall-clock time from ResultsWriter construction to this cell, in seconds.")
print(f"total_runtime_seconds so far: {total_runtime_seconds:.1f}s")
runtime_budget.groupby("phase")["elapsed_seconds"].sum()

total_runtime_seconds so far: 321.5s


phase
binary_refit      1.490578
final_fit       198.839490
oof_cv            1.024618
search           34.936664
Name: elapsed_seconds, dtype: float64

In [33]:
writer.add_note(
    "single_test_evaluation",
    "The cleaned testing partition is scored exactly once per model per TTL variant (test_evaluations_count=1); no test-informed model choice, hyperparameter, or threshold decision precedes that single pass.",
)
writer.add_note(
    "subsample_trained_models",
    f"GaussianProcessClassifier trains on at most {GP_MAX_ROWS} rows, SVC(RBF) on at most {SVC_MAX_ROWS} rows, and KNeighborsClassifier references at most {KNN_REF_ROWS} rows; every table flags training_rows and subsampled so these are never compared as equals to the {len(cleaned.train)}-row full-data models.",
)
writer.add_note(
    "label_noise_error_floor",
    "The retained testing set carries feature-identical, label-contradictory rows inherited from the raw UNSW-NB15 partitions; this is a near-certain minimum error rate no classifier can eliminate, reported beside the accuracy results rather than folded into them.",
)
writer.add_note(
    "ttl_shortcut_comparison",
    "Every model is searched, refit and evaluated twice: once with the sttl/ct_state_ttl testbed-shortcut pair and once without, so model_comparison.csv exposes exactly how much of each model's score the shortcut pair contributes.",
)
writer.add_note(
    "cost_threshold_assumption",
    f"FN:FP = {COST_FN:.0f}:{COST_FP:.0f}. A missed intrusion carries incident-response and breach cost roughly an order of magnitude above one analyst triaging a false alarm; this ratio is a deliberately conservative SOC assumption, not a measurement.",
)
writer.add_note(
    "benchmark_non_comparability",
    "GaussianProcessClassifier, SVC and KNeighborsClassifier are trained or referenced on a capped subsample while DummyClassifier, LogisticRegression, RandomForestClassifier and HistGradientBoostingClassifier train on the full training allocation; comparing their scores as if trained on equal data would be misleading, which is why training_rows and subsampled accompany every ranking table.",
)
writer.add_note(
    "soc_conclusions",
    "See the markdown conclusions cell below for what a SOC would deploy, what the TTL delta implies about the testbed, what the label-noise floor implies for expected alert quality, and what the subsample caps do and do not let this notebook claim.",
)
print(f"{len(writer.notes)} notes registered.")

7 notes registered.


## Conclusions for a SOC

**What to deploy.** Rank models by macro F1, not accuracy — `Normal` outnumbers `Worms` by
roughly 400 to 1, so a headline accuracy score hides how a model treats the rarest attack
classes. `model_comparison.csv` and the per-class tables above show the actual trade-off between
the top-ranked model and its false-negative/false-positive behaviour on rare classes.

**What the TTL delta means.** `sttl` and `ct_state_ttl` are known artefacts of how the UNSW-NB15
testbed generated traffic, not features a real network necessarily exposes the same way. A large
`macro_f1_delta` in `model_comparison.csv` means a meaningful share of the reported score depends
on a testbed shortcut rather than genuine attack signal — the `without_ttl` numbers are the more
honest estimate of what a model trained on unfamiliar traffic would achieve.

**What the error floor means.** `label_noise_floor.csv`'s `test_contradictory_share_of_retained`
is a hard lower bound on achievable error: this fraction of the testing set cannot be classified
correctly by any model, because two feature-identical rows in the source data carry different
labels. Any reported error rate should be read relative to this floor, not to zero.

**What the subsample caps do and do not let us claim.** `GaussianProcessClassifier`, `SVC(RBF)`,
and `KNeighborsClassifier` are trained or referenced on a capped subsample (`subsampled=True` in
every ranking table). Their scores are valid measurements of what those algorithms achieve *on
that subsample size* — they are not evidence of what those algorithms would achieve if trained on
the full 107,740-row allocation, and should not be compared to the full-data models as if they
were.

In [34]:
print("=== Final summary ===")
print(f"Best with_ttl model:    {best_model_with_ttl} (macro F1 = {best_row_with_ttl['macro_f1']:.4f}, balanced accuracy = {best_row_with_ttl['balanced_accuracy']:.4f})")
print(f"Best without_ttl model: {best_model_without_ttl} (macro F1 = {best_row_without_ttl['macro_f1']:.4f}, balanced accuracy = {best_row_without_ttl['balanced_accuracy']:.4f})")
print(f"Dummy baseline macro F1 (with_ttl): {dummy_macro_f1:.4f}")
print(f"Binary ROC-AUC: with_ttl={BINARY_ROC_AUC['with_ttl']:.4f} without_ttl={BINARY_ROC_AUC['without_ttl']:.4f}")
print(f"Binary AP:      with_ttl={BINARY_AVERAGE_PRECISION['with_ttl']:.4f} without_ttl={BINARY_AVERAGE_PRECISION['without_ttl']:.4f}")
print(f"Cost-optimal thresholds: {COST_OPTIMAL_THRESHOLD}")
print(f"Label-noise floor: {LABEL_NOISE_FLOOR}")
model_comparison.sort_values("macro_f1_with_ttl", ascending=False)

=== Final summary ===
Best with_ttl model:    hist_gradient_boosting (macro F1 = 0.5116, balanced accuracy = 0.5768)
Best without_ttl model: hist_gradient_boosting (macro F1 = 0.5029, balanced accuracy = 0.5529)
Dummy baseline macro F1 (with_ttl): 0.0893
Binary ROC-AUC: with_ttl=0.9829 without_ttl=0.9830
Binary AP:      with_ttl=0.9871 without_ttl=0.9871
Cost-optimal thresholds: {'with_ttl': 0.075, 'without_ttl': 0.08499999999999999}
Label-noise floor: {'available': True, 'rows': 4294, 'share': 0.05499205983300036}


,model,training_rows,subsampled,accuracy_with_ttl,balanced_accuracy_with_ttl,macro_f1_with_ttl,weighted_f1_with_ttl,accuracy_without_ttl,balanced_accuracy_without_ttl,macro_f1_without_ttl,weighted_f1_without_ttl,macro_f1_delta
4,hist_gradient_boosting,107740,False,0.742969,0.576820,0.511635,0.774731,0.741522,0.552864,0.502885,0.774130,0.008750
3,random_forest,107740,False,0.691857,0.638170,0.502626,0.743445,0.690321,0.628490,0.493598,0.741380,0.009028
2,knn,20000,True,0.706137,0.456552,0.420246,0.736642,0.697300,0.442625,0.395119,0.728389,0.025127
5,svc,5000,True,0.618693,0.579629,0.376698,0.679069,0.610599,0.522034,0.382057,0.663550,-0.005359
6,gaussian_process,2000,True,0.676297,0.461545,0.372736,0.710582,0.667333,0.440046,0.357932,0.700448,0.014805
1,logistic_regression,107740,False,0.614479,0.577193,0.356204,0.679248,0.606744,0.571999,0.352012,0.672434,0.004192
0,dummy,107740,False,0.268800,0.097858,0.089319,0.261649,0.268800,0.097858,0.089319,0.261649,0.000000


In [35]:
closed_dir = writer.close()
print(f"Results written to {closed_dir}")

Results written to /home/pato/Desktop/machine_learning_project/results/classification


In [36]:
# Assertion 3: manifest round-trip. Every declared path exists; every file on disk is declared.
manifest_path = closed_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
declared_paths = {entry["path"] for entry in manifest["tables"] + manifest["figures"]}

for path in declared_paths:
    assert (closed_dir / path).is_file(), f"Manifest declares missing file: {path}"

on_disk = set()
for sub in ("tables", "figures"):
    for file in (closed_dir / sub).iterdir():
        if file.is_file() and file.suffix in (".csv", ".json", ".png"):
            on_disk.add(f"{sub}/{file.name}")

assert on_disk == declared_paths, f"Undeclared or missing files: {on_disk.symmetric_difference(declared_paths)}"
print(f"Assertion 3 passed: manifest declares exactly {len(declared_paths)} files, all present on disk.")
print(f"tables={len(manifest['tables'])} figures={len(manifest['figures'])} metrics={len(manifest['metrics'])} notes={len(manifest['notes'])}")

Assertion 3 passed: manifest declares exactly 26 files, all present on disk.
tables=17 figures=9 metrics=30 notes=7


## Reproducibility and limits

- Every split, subsample, search and model uses `random_state=42` / `seed=42`.
- Re-running this notebook end to end over identical cleaned inputs reproduces every `tables/`
  file byte-for-byte (deterministic subsampling, deterministic CV, fixed hyperparameter grids).
- Search hyperparameters are chosen on a 30,000-row train subsample, not the full training
  allocation — the winner is refit on its full allocation, but the *search* itself saw a smaller
  slice of the training distribution.
- `GaussianProcessClassifier`, `SVC(RBF)` and `KNeighborsClassifier` are capped below the full
  training allocation for compute-budget reasons (see `runtime_budget.csv` and the
  `subsample_trained_models` note); their scores are not directly comparable to the full-data
  models.
- The cost ratio behind the binary threshold (FN:FP = 20:1) is a stated SOC assumption, not a
  measurement — a different organisation's true cost ratio would move the threshold.
- This notebook does not claim to have found the causal reason any model performs better or
  worse; it reports what was measured, on the partitions and caps declared above.